# Baseline ResNet-50 Retraining and Additional Evaluation

This notebook reproduces the original ImageNet-pretrained ResNet-50 baseline experiment
with the saved patient-wise five-fold assignments and independent 15%
held-out test set. It does not tune any setting on the test set.

The additional outputs address dataset-source-specific performance,
confusion matrices, paired patient-clustered uncertainty, ROC/PR
performance, and probability calibration. XAI plausibility and
fidelity experiments are outside this notebook's scope.

## 1. Environment and Configuration

In [ ]:
import platform
import sys

import torch

print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path

ORIGINAL_EXPERIMENT_ID = "fa028196-fd4c-441f-a5d3-3efb9707e7f9"
RETRAINING_EXPERIMENT_ID = (
    "fa028196-fd4c-441f-a5d3-3efb9707e7f9-retraining-01"
)
EXPERIMENT_COMPONENT = "classification/cv_resnet50"
COUNTERPART_COMPONENT = "classification/multistage_direct_guided_cv_resnet50"
MODEL_NAME = "Baseline"
ORIGINAL_CONFIG_SNAPSHOT = "cv_resnet50.json"

REPOSITORY_URL = (
    "https://github.com/FillipusAditya/"
    "mask-guided-lung-nodule-xai.git"
)
REPOSITORY_BRANCH = "refactor/002-segmentation"
PROJECT_ROOT = Path("/content/mask-guided-lung-nodule-xai")
DRIVE_PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mask-guided-lung-nodule-xai"
)
DRIVE_OUTPUT_ROOT = DRIVE_PROJECT_ROOT / "experiment_results"
ORIGINAL_EXPERIMENT_ROOT = (
    DRIVE_OUTPUT_ROOT / ORIGINAL_EXPERIMENT_ID
)
ORIGINAL_RESULT_DIR = (
    ORIGINAL_EXPERIMENT_ROOT / EXPERIMENT_COMPONENT
)
RETRAINING_EXPERIMENT_ROOT = (
    DRIVE_OUTPUT_ROOT / RETRAINING_EXPERIMENT_ID
)
DRIVE_OUTPUT_DIR = (
    RETRAINING_EXPERIMENT_ROOT / EXPERIMENT_COMPONENT
)
COUNTERPART_OUTPUT_DIR = (
    RETRAINING_EXPERIMENT_ROOT / COUNTERPART_COMPONENT
)

ARCHIVE_NAME = (
    f"{ORIGINAL_EXPERIMENT_ID}_guided_resnet50_parenchyma.tar.gz"
)
DRIVE_ARCHIVE_PATH = (
    ORIGINAL_EXPERIMENT_ROOT / ARCHIVE_NAME
)
DRIVE_CHECKSUM_PATH = Path(str(DRIVE_ARCHIVE_PATH) + ".sha256")
LOCAL_ARCHIVE_PATH = Path("/content") / ARCHIVE_NAME
EXTRACTION_ROOT = (
    Path("/content") / f"retraining_data_{ORIGINAL_EXPERIMENT_ID}"
)
DATASET_ROOT = EXTRACTION_ROOT / "000_dataset_v2/_segmentation_dataset"
METADATA_PATH = DATASET_ROOT / "004_classification_cv_5fold_seed42.csv"
HOLDOUT_METADATA_PATH = DATASET_ROOT / "001_holdout_split_lidc_lndb.csv"
PROBABILITY_ROOT = (
    EXTRACTION_ROOT
    / "experiment_results"
    / ORIGINAL_EXPERIMENT_ID
    / "segmentation/unet/inference/probability_npy"
)

COPY_ARCHIVE_TO_LOCAL = True
VERIFY_ARCHIVE_SHA256 = True
FORCE_EXTRACT = False
RUN_TRAINING = True
RUN_HELD_OUT_INFERENCE = True
RUN_ADDITIONAL_EVALUATION = True
TEST_BATCH_SIZE = 1
TEST_NUM_WORKERS = 0
FIGURE_DPI = 500
CALIBRATION_BINS = 10
BOOTSTRAP_ITERATIONS = 10_000
BOOTSTRAP_SEED = 42
DEVICE = "auto"

print(f"Original results : {ORIGINAL_RESULT_DIR}")
print(f"Archive          : {DRIVE_ARCHIVE_PATH}")
print(f"Retraining output: {DRIVE_OUTPUT_DIR}")

### 1.1 Repository and dependencies

In [ ]:
import subprocess

if (PROJECT_ROOT / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "fetch", "origin"],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "checkout", REPOSITORY_BRANCH],
        check=True,
    )
    subprocess.run(
        [
            "git", "-C", str(PROJECT_ROOT), "pull", "--ff-only",
            "origin", REPOSITORY_BRANCH,
        ],
        check=True,
    )
else:
    subprocess.run(
        [
            "git", "clone", "--branch", REPOSITORY_BRANCH,
            "--single-branch", REPOSITORY_URL, str(PROJECT_ROOT),
        ],
        check=True,
    )

print(f"Repository ready: {PROJECT_ROOT}")

In [ ]:
packages = [
    "albumentations>=2.0,<3.0",
    "opencv-python-headless",
    "pandas",
    "matplotlib",
    "scikit-learn",
    "tqdm",
    "zennit==0.5.1",
    "Pillow",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *packages],
    check=True,
)
print("Dependencies ready.")

## 2. Dataset Preparation

The archive is copied and verified before extraction. It already
contains all U-Net probability maps required by the proposed model;
U-Net inference and U-Net retraining are not performed here.

In [ ]:
import hashlib
import hmac
import shutil
import tarfile

from tqdm.auto import tqdm


def copy_with_progress(source, destination, chunk_size=8 * 1024 * 1024):
    """Copy one file while reporting byte-level progress."""

    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".part")
    with source.open("rb") as input_file, temporary.open("wb") as output_file:
        with tqdm(
            total=source.stat().st_size,
            desc="Copying archive",
            unit="B",
            unit_scale=True,
        ) as progress:
            while True:
                chunk = input_file.read(chunk_size)
                if not chunk:
                    break
                output_file.write(chunk)
                progress.update(len(chunk))
    temporary.replace(destination)


def calculate_sha256(path, chunk_size=8 * 1024 * 1024):
    """Calculate the SHA-256 digest of one file."""

    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


if not DRIVE_ARCHIVE_PATH.is_file():
    raise FileNotFoundError(f"Archive not found: {DRIVE_ARCHIVE_PATH}")
if VERIFY_ARCHIVE_SHA256 and not DRIVE_CHECKSUM_PATH.is_file():
    raise FileNotFoundError(f"Checksum not found: {DRIVE_CHECKSUM_PATH}")

archive_path = DRIVE_ARCHIVE_PATH
if COPY_ARCHIVE_TO_LOCAL:
    archive_path = LOCAL_ARCHIVE_PATH
    current = (
        archive_path.is_file()
        and archive_path.stat().st_size == DRIVE_ARCHIVE_PATH.stat().st_size
    )
    if not current:
        copy_with_progress(DRIVE_ARCHIVE_PATH, archive_path)

archive_digest = calculate_sha256(archive_path)
if VERIFY_ARCHIVE_SHA256:
    expected_digest = (
        DRIVE_CHECKSUM_PATH.read_text(encoding="utf-8")
        .split()[0]
        .lower()
    )
    if not hmac.compare_digest(archive_digest, expected_digest):
        raise RuntimeError("Archive SHA-256 verification failed.")

marker = EXTRACTION_ROOT / f".extracted_{archive_digest[:16]}"
if FORCE_EXTRACT and EXTRACTION_ROOT.exists():
    shutil.rmtree(EXTRACTION_ROOT)
if not marker.is_file():
    EXTRACTION_ROOT.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path, mode="r:gz") as archive:
        members = archive.getmembers()
        for member in tqdm(members, desc="Extracting archive", unit="file"):
            archive.extract(member, path=EXTRACTION_ROOT, filter="data")
    marker.touch()

print(f"Archive SHA-256: {archive_digest}")
print(f"Extraction root: {EXTRACTION_ROOT}")

### 2.1 Metadata, files, labels, and probability-map validation

In [ ]:
import pandas as pd

metadata = pd.read_csv(METADATA_PATH)
required_columns = {
    "dataset", "patient_id", "filename", "ct_parenchyma_path",
    "mask_path", "label", "cv_group_id", "cv_nodule_id",
    "cv_role", "cv_fold",
}
missing_columns = required_columns - set(metadata.columns)
if missing_columns:
    raise ValueError(f"Missing metadata columns: {sorted(missing_columns)}")
if not metadata["label"].str.lower().isin(["benign", "malignant"]).all():
    raise ValueError("Metadata contains invalid labels.")
if metadata[["dataset", "filename"]].duplicated().any():
    raise ValueError("Metadata contains duplicate dataset/filename rows.")

missing_files = []
for row in tqdm(
    metadata.itertuples(index=False),
    total=len(metadata),
    desc="Validating classification files",
    unit="slice",
):
    for relative_path in (row.ct_parenchyma_path, row.mask_path):
        path = DATASET_ROOT / relative_path
        if not path.is_file():
            missing_files.append(path)
if missing_files:
    raise FileNotFoundError(
        f"Missing {len(missing_files)} CT or mask files; "
        f"examples: {missing_files[:5]}"
    )

expected_probability_names = set(metadata["filename"])
available_probability_names = {
    path.name for path in PROBABILITY_ROOT.glob("*.npy")
}
missing_probability_maps = (
    expected_probability_names - available_probability_names
)
unexpected_probability_maps = (
    available_probability_names - expected_probability_names
)
if missing_probability_maps or unexpected_probability_maps:
    raise RuntimeError(
        "Probability-map filenames do not match classification metadata: "
        f"missing={len(missing_probability_maps)}, "
        f"unexpected={len(unexpected_probability_maps)}"
    )

print(f"Metadata slices: {len(metadata):,}")
print(f"Probability maps: {len(available_probability_names):,}")
print("All CT, mask, and probability-map filenames are valid.")

## 3. Reproducibility and Fold Validation

The original 85% development cohort and 15% held-out test cohort are
preserved. Fold assignments are reconstructed with
`StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)`, using
dataset source plus patient ID for grouping and dataset source plus
nodule label for stratification. The reconstructed assignments must
exactly match the saved metadata.

In [ ]:
import importlib

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

cv_module = importlib.import_module(
    "001_7_dataset_split.dataset_split.cross_validation"
)
holdout_metadata = cv_module.load_metadata(HOLDOUT_METADATA_PATH)
identified_metadata = cv_module.add_identifiers(holdout_metadata)
cv_module.validate_source_membership(identified_metadata)
nodule_metadata = cv_module.build_nodule_metadata(identified_metadata)
assigned_nodules = cv_module.assign_nodule_folds(
    nodule_metadata,
    n_splits=5,
    random_seed=42,
    shuffle=True,
)
reconstructed = cv_module.add_slice_fold_assignments(
    identified_metadata,
    assigned_nodules,
)

comparison_columns = [
    "dataset", "filename", "cv_group_id", "cv_nodule_id",
    "cv_role", "cv_fold",
]
expected = metadata[comparison_columns].copy()
observed = reconstructed[comparison_columns].copy()
expected["cv_role"] = expected["cv_role"].astype(str)
observed["cv_role"] = observed["cv_role"].astype(str)
expected = expected.sort_values(["dataset", "filename"]).reset_index(drop=True)
observed = observed.sort_values(["dataset", "filename"]).reset_index(drop=True)
if not expected.equals(observed):
    raise RuntimeError("Reconstructed fold assignments do not match metadata.")

development = metadata.loc[metadata["cv_role"].eq("development")]
holdout = metadata.loc[metadata["cv_role"].eq("holdout_test")]
if set(development["cv_group_id"]) & set(holdout["cv_group_id"]):
    raise RuntimeError("Patient overlap exists between development and test.")
for fold in range(5):
    training = development.loc[development["cv_fold"].ne(fold)]
    validation = development.loc[development["cv_fold"].eq(fold)]
    for column in ("cv_group_id", "cv_nodule_id", "filename"):
        overlap = set(training[column]) & set(validation[column])
        if overlap:
            raise RuntimeError(
                f"Fold {fold} has overlap in {column}: {len(overlap)}"
            )

print("Saved and reconstructed fold assignments match exactly.")
print("No patient, nodule, or slice overlap was found in any fold.")
print(f"Development patients: {development['cv_group_id'].nunique():,}")
print(f"Held-out patients: {holdout['cv_group_id'].nunique():,}")

### 3.1 Load the original experiment configuration

In [ ]:
import copy
import json

original_config_path = ORIGINAL_RESULT_DIR / ORIGINAL_CONFIG_SNAPSHOT
if not original_config_path.is_file():
    raise FileNotFoundError(
        f"Original configuration not found: {original_config_path}"
    )
with original_config_path.open("r", encoding="utf-8") as stream:
    original_config = json.load(stream)

original_fold_artifacts = []
for fold in range(5):
    fold_dir = ORIGINAL_RESULT_DIR / f"fold_{fold}"
    original_fold_artifacts.append(fold_dir)
    if not fold_dir.is_dir():
        raise FileNotFoundError(f"Original fold directory missing: {fold_dir}")
    history_path = fold_dir / "training_log.csv"
    checkpoint_path = fold_dir / "best_model.pth"
    if not history_path.is_file() or not checkpoint_path.is_file():
        raise FileNotFoundError(
            f"Original fold artifacts are incomplete: {fold_dir}"
        )
    history = pd.read_csv(history_path)
    if history.empty or not {"train_loss", "val_loss"}.issubset(history):
        raise ValueError(f"Invalid original training log: {history_path}")

if int(original_config["training"]["num_epochs"]) != 200:
    raise RuntimeError("Original maximum epochs are not 200.")
if int(original_config["training"]["batch_size"]) != 64:
    raise RuntimeError("Original batch size is not 64.")
if float(original_config["training"]["learning_rate"]) != 1e-3:
    raise RuntimeError("Original learning rate is not 1e-3.")
if int(original_config["training"]["seed"]) != 42:
    raise RuntimeError("Original seed is not 42.")
if int(original_config["cross_validation"]["num_folds"]) != 5:
    raise RuntimeError("Original experiment does not use five folds.")

config = copy.deepcopy(original_config)
config["experiment"]["id"] = RETRAINING_EXPERIMENT_ID
config["output"]["root_directory"] = str(DRIVE_OUTPUT_ROOT)
config["data"]["dataset_root"] = str(DATASET_ROOT)
config["data"]["metadata_path"] = str(METADATA_PATH)
config["training"]["device"] = DEVICE

In [ ]:
if config["model"]["architecture"] != "ResNet50":
    raise RuntimeError("Unexpected baseline architecture.")
if bool(config["scheduler"]["enabled"]):
    raise RuntimeError("The original baseline scheduler was disabled.")

In [ ]:
CONFIG_PATH = (
    PROJECT_ROOT / "003_classification/configs/cv_resnet50.json"
)
SAVED_CONFIG_PATH = (
    DRIVE_OUTPUT_ROOT / "saved_configs" / RETRAINING_EXPERIMENT_ID
    / "cv_resnet50.json"
)


def save_json(values, path):
    """Save a dictionary as readable JSON."""

    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as stream:
        json.dump(values, stream, indent=4)
        stream.write("\n")


save_json(config, CONFIG_PATH)
save_json(config, SAVED_CONFIG_PATH)
print(json.dumps(config, indent=2))
print(f"Retraining configuration: {CONFIG_PATH}")

## 4. Model Initialization

In [ ]:
importlib.invalidate_caches()
module_name = "003_classification.cv_resnet50.train"
if module_name in sys.modules:
    del sys.modules[module_name]
train_module = importlib.import_module(module_name)

train_loader, validation_loader, _, _ = (
    train_module.build_fold_dataloaders(fold=0)
)
model = train_module.build_model(num_classes=2)
trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)
total_parameters = sum(parameter.numel() for parameter in model.parameters())
if trainable_parameters != total_parameters:
    raise RuntimeError("The full baseline model is not trainable.")
print(f"Fold 0 training slices: {len(train_loader.dataset):,}")
print(f"Fold 0 validation slices: {len(validation_loader.dataset):,}")
print(f"Trainable parameters: {trainable_parameters:,}")
del model, train_loader, validation_loader
torch.cuda.empty_cache()

## 5. Five-Fold Cross-Validation Training

In [ ]:
if RUN_TRAINING:
    if DRIVE_OUTPUT_DIR.exists():
        raise FileExistsError(
            f"Retraining output already exists: {DRIVE_OUTPUT_DIR}"
        )
    train_module.main()
elif not DRIVE_OUTPUT_DIR.is_dir():
    raise FileNotFoundError(
        f"Saved retraining output not found: {DRIVE_OUTPUT_DIR}"
    )

## 6. Checkpoint Selection

In [ ]:
required_training_outputs = [DRIVE_OUTPUT_DIR / "cv_summary.csv"]
for fold in range(5):
    required_training_outputs.extend(
        [
            DRIVE_OUTPUT_DIR / f"fold_{fold}/best_model.pth",
            DRIVE_OUTPUT_DIR / f"fold_{fold}/checkpoint_latest.pth",
            DRIVE_OUTPUT_DIR / f"fold_{fold}/training_log.csv",
        ]
    )
missing = [path for path in required_training_outputs if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Incomplete training artifacts: {missing[:10]}")

summary = pd.read_csv(DRIVE_OUTPUT_DIR / "cv_summary.csv")
if set(summary["fold"].astype(int)) != set(range(5)):
    raise RuntimeError("Training summary does not contain all five folds.")
display(summary)
print("Every fold has a best-validation-loss checkpoint.")

## 7. Held-Out Test Inference

Each held-out slice is evaluated by the five best fold checkpoints.
Their softmax probabilities are averaged, and malignant probability
is thresholded at 0.5. The same ensemble predictions are reused for
combined and dataset-source-specific evaluation.

In [ ]:
if RUN_HELD_OUT_INFERENCE:
    test_output = DRIVE_OUTPUT_DIR / "test"
    if test_output.exists():
        raise FileExistsError(f"Test output already exists: {test_output}")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "003_classification.cv_resnet50.test",
            str(DRIVE_OUTPUT_DIR),
            "--dataset-root", str(DATASET_ROOT),
            "--metadata-path", str(METADATA_PATH),
            "--batch-size", str(TEST_BATCH_SIZE),
            "--num-workers", str(TEST_NUM_WORKERS),
            "--device", DEVICE,
            "--dpi", str(FIGURE_DPI),
            "--force-inference",
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )

In [ ]:
test_predictions_path = DRIVE_OUTPUT_DIR / "test/test_predictions.csv"
if not test_predictions_path.is_file():
    raise FileNotFoundError(
        f"Held-out predictions not found: {test_predictions_path}"
    )
test_predictions = pd.read_csv(test_predictions_path)
if not test_predictions["cv_role"].eq("holdout_test").all():
    raise RuntimeError("Non-held-out samples were found in test predictions.")
if test_predictions["cv_group_id"].isin(development["cv_group_id"]).any():
    raise RuntimeError("A development patient was found in test predictions.")
print(f"Held-out ensemble slices: {len(test_predictions):,}")

## 8. Nodule-Level Majority Voting

The primary nodule prediction uses majority voting across slice-level
labels. Exact vote ties are resolved with the mean malignant
probability at the unchanged 0.5 threshold. Mean slice probability is
retained as the continuous nodule score for ROC/PR and calibration.

In [ ]:
voting_module = importlib.import_module(
    "004_evaluation.classification_nodule_voting"
)
voting_module.write_model_outputs(
    model_dir=DRIVE_OUTPUT_DIR,
    validation_source=DRIVE_OUTPUT_DIR / "out_of_fold_predictions.csv",
    test_source=test_predictions_path,
)

## 9. Source-Specific Evaluation

In [ ]:
evaluation_module = importlib.import_module(
    "004_evaluation.retraining_classification_evaluation"
)
if RUN_ADDITIONAL_EVALUATION:
    evaluation_outputs = evaluation_module.run_single_model_evaluation(
        model_dir=DRIVE_OUTPUT_DIR,
        model_name=MODEL_NAME,
        metadata_path=METADATA_PATH,
        calibration_bins=CALIBRATION_BINS,
        dpi=FIGURE_DPI,
    )
    for name, path in evaluation_outputs.items():
        print(f"{name}: {path}")

## 10. Statistical Evaluation

Paired patient-clustered bootstrap is performed at nodule level with
10,000 iterations, a 95% percentile interval, seed 42, and identical
resampled patients for both models. All nodules from every selected
patient are included. Undefined class-specific metrics remain missing
rather than being replaced by zero.

In [ ]:
comparison_output_dir = (
    RETRAINING_EXPERIMENT_ROOT
    / "evaluation/baseline_vs_proposed"
)
required_counterpart_outputs = [
    COUNTERPART_OUTPUT_DIR / "test/test_predictions.csv",
    COUNTERPART_OUTPUT_DIR
    / "nodule_voting/test_nodule_predictions.csv",
]
current_outputs = [
    DRIVE_OUTPUT_DIR / "test/test_predictions.csv",
    DRIVE_OUTPUT_DIR / "nodule_voting/test_nodule_predictions.csv",
]
if all(path.is_file() for path in current_outputs + required_counterpart_outputs):
    baseline_dir = (
        DRIVE_OUTPUT_DIR if MODEL_NAME == "Baseline"
        else COUNTERPART_OUTPUT_DIR
    )
    proposed_dir = (
        DRIVE_OUTPUT_DIR if MODEL_NAME == "Proposed"
        else COUNTERPART_OUTPUT_DIR
    )
    paired_outputs = evaluation_module.run_paired_comparison(
        baseline_dir=baseline_dir,
        proposed_dir=proposed_dir,
        output_dir=comparison_output_dir,
        bootstrap_iterations=BOOTSTRAP_ITERATIONS,
        seed=BOOTSTRAP_SEED,
        dpi=FIGURE_DPI,
    )
    for name, path in paired_outputs.items():
        print(f"{name}: {path}")
else:
    print(
        "Paired comparison is deferred until both retraining notebooks "
        "have produced slice- and nodule-level test predictions."
    )

## 11. Reproduction Difference Report

In [ ]:
original_prediction_paths = {
    "slice": ORIGINAL_RESULT_DIR / "test/test_predictions.csv",
    "nodule": (
        ORIGINAL_RESULT_DIR
        / "nodule_voting/test_nodule_predictions.csv"
    ),
}
comparison_frames = []
retraining_metrics = pd.read_csv(
    DRIVE_OUTPUT_DIR
    / "source_specific_evaluation/source_specific_metrics.csv"
)
for level, path in original_prediction_paths.items():
    original_predictions = evaluation_module.prepare_predictions(path, level)
    original_metrics = evaluation_module.source_metric_table(
        original_predictions,
        MODEL_NAME,
        level,
    )
    current = retraining_metrics.loc[
        retraining_metrics["evaluation_level"].eq(level)
    ]
    merged = original_metrics.merge(
        current,
        on=["model", "dataset_scope", "evaluation_level"],
        suffixes=("_original", "_retraining"),
        validate="one_to_one",
    )
    for metric in evaluation_module.METRIC_NAMES:
        merged[f"difference_{metric}"] = (
            merged[f"{metric}_retraining"]
            - merged[f"{metric}_original"]
        )
    comparison_frames.append(merged)

reproduction_report = pd.concat(comparison_frames, ignore_index=True)
reproduction_path = (
    DRIVE_OUTPUT_DIR
    / "source_specific_evaluation/reproduction_difference_report.csv"
)
reproduction_report.to_csv(reproduction_path, index=False)
display(reproduction_report)
print(f"Reproduction report: {reproduction_path}")

## 12. Results Visualization and Export

In [ ]:
from IPython.display import Image, display

source_metrics_path = (
    DRIVE_OUTPUT_DIR
    / "source_specific_evaluation/source_specific_metrics.csv"
)
calibration_summary_path = (
    DRIVE_OUTPUT_DIR / "calibration/calibration_summary.csv"
)
roc_pr_summary_path = (
    DRIVE_OUTPUT_DIR
    / "roc_pr_evaluation/roc_auc_average_precision_summary.csv"
)
display(pd.read_csv(source_metrics_path))
display(pd.read_csv(roc_pr_summary_path))
display(pd.read_csv(calibration_summary_path))

preview_path = (
    DRIVE_OUTPUT_DIR
    / "confusion_matrices/nodule_combined_confusion_matrix.png"
)
display(Image(filename=str(preview_path), width=500))

## 13. Saved Artifact Validation

In [ ]:
from PIL import Image as PILImage

required_outputs = [
    DRIVE_OUTPUT_DIR / "test/test_predictions.csv",
    DRIVE_OUTPUT_DIR / "test/test_results.json",
    DRIVE_OUTPUT_DIR / "nodule_voting/test_nodule_predictions.csv",
    DRIVE_OUTPUT_DIR / "nodule_voting/test_nodule_metrics.csv",
    DRIVE_OUTPUT_DIR
    / "source_specific_evaluation/source_specific_metrics.csv",
    DRIVE_OUTPUT_DIR
    / "source_specific_evaluation/source_class_split_distribution.csv",
    DRIVE_OUTPUT_DIR
    / "confusion_matrices/classification_metrics_summary.csv",
    DRIVE_OUTPUT_DIR
    / "roc_pr_evaluation/roc_auc_average_precision_summary.csv",
    DRIVE_OUTPUT_DIR / "calibration/calibration_summary.csv",
]
missing_outputs = [path for path in required_outputs if not path.is_file()]
if missing_outputs:
    raise FileNotFoundError(
        f"Required outputs are missing: {missing_outputs}"
    )

low_dpi_figures = []
for path in DRIVE_OUTPUT_DIR.rglob("*.png"):
    with PILImage.open(path) as image:
        dpi = image.info.get("dpi")
    if dpi is not None and min(dpi) < 490:
        low_dpi_figures.append((path, dpi))
if low_dpi_figures:
    raise RuntimeError(
        f"Figures below 500-DPI export tolerance: {low_dpi_figures[:5]}"
    )

print(f"Validated output directory: {DRIVE_OUTPUT_DIR}")
print(f"PNG files checked: {len(list(DRIVE_OUTPUT_DIR.rglob('*.png'))):,}")
print("All required retraining and evaluation artifacts are present.")

## 14. Rerunning Evaluation Without Retraining

Set `RUN_TRAINING = False` and `RUN_HELD_OUT_INFERENCE = False` in
Section 1 when checkpoints and cached prediction CSV files already
exist. Sections 8-13 can then reproduce nodule voting, source-specific
metrics, statistical summaries, and figures without classifier
retraining or repeated held-out inference.